# Try PyGeoParallax-IN

[PyGeoParallax-IN](https://github.com/bapallinaveen60/PyGeoParallax-IN) corrects INSAT-3DR satellite images for **cloud parallax**. The satellite sees tall clouds at a slant, so they appear displaced away from it, by up to about 10 km over Northeast India. The tool moves each cloudy pixel back to where the cloud really is.

This notebook runs in your browser; nothing is installed on your computer. Run the cells from the top (**Runtime → Run all**):

1. install the package (about 1 minute);
2. run the built-in demo, a synthetic scene with storms at known positions, so you can see the correction work with no data at all;
3. optionally, correct one of your own INSAT-3DR L1C files.

Problems or questions: [open an issue](https://github.com/bapallinaveen60/PyGeoParallax-IN/issues).

In [ ]:
!pip install -q git+https://github.com/bapallinaveen60/PyGeoParallax-IN.git
!pygeoparallax --version

## 1. Demo: storms at known positions

Three storms with flat tops at 16, 12 and 8 km are placed at known positions over Northeast India. The demo simulates exactly what INSAT-3DR (at 74° E) would see, corrects the image the normal way (cloud height from the brightness temperature), and checks that every storm ends up within one pixel (4 km) of its true position.

In [ ]:
from IPython.display import Image, display
from pygeoparallax.demo import run_demo

result = run_demo("demo")
display(Image("demo/demo.png"))

The red circles are the true storm positions. In the left panel the storms are shifted north-east, away from the satellite, by more the higher they are. After correction (middle) they sit on their true positions. The right panel shows how far each pixel was moved.

## 2. Your own INSAT-3DR file (optional)

Download an INSAT-3DR Imager L1C sector file (`3RIMG_<date>_<time>_L1C_SGP_V01R00.h5`, about 90 MB) from [MOSDAC](https://www.mosdac.gov.in) (free registration). Then run the next cell and choose the file in the upload box.

`BBOX` limits the correction to a region (latitude and longitude limits), which keeps it fast. The default covers Northeast India and Bangladesh.

In [ ]:
import glob, os

BBOX = (20.0, 30.0, 85.0, 97.0)   # lat_min, lat_max, lon_min, lon_max

try:
    from google.colab import files
    uploaded = list(files.upload())
except ImportError:              # not in Colab: use L1C files already in this folder
    uploaded = []
l1c = (uploaded or sorted(glob.glob("3RIMG_*L1C*.h5")))[:1]
print("file:", l1c[0] if l1c else "none (skip this section)")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

from pygeoparallax.cli import process_l1c

if l1c:
    name = os.path.basename(l1c[0]).replace(".h5", "")
    process_l1c(l1c[0], f"out/{name}_uncorrected.nc", bbox=BBOX, native=True, correct=False)
    process_l1c(l1c[0], f"out/{name}_corrected.nc", bbox=BBOX, native=True, fill_empty=True)
    raw = xr.open_dataset(f"out/{name}_uncorrected.nc")
    cor = xr.open_dataset(f"out/{name}_corrected.nc")
    cloudy = np.isfinite(cor["cth"].values)
    print(f"{cor.attrs['satellite']} {cor.attrs['time']}: cloudy {cloudy.mean():.0%}, "
          f"median shift of cloudy pixels {np.nanmedian(cor['shift_km'].values[cloudy]):.1f} km")

    fig, ax = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
    lat, lon = cor["lat"].values, cor["lon"].values
    for a, ds, t in ((ax[0], raw, "uncorrected"), (ax[1], cor, "parallax-corrected")):
        im = a.pcolormesh(lon, lat, ds["TIR1"], cmap="Greys", vmin=190, vmax=300, shading="auto")
        a.set_title(f"TIR1, {t}")
    fig.colorbar(im, ax=ax[:2], label="brightness temperature [K]", shrink=0.9)
    im = ax[2].pcolormesh(lon, lat, cor["shift_km"].where(cloudy), cmap="viridis", shading="auto")
    fig.colorbar(im, ax=ax[2], label="shift [km]", shrink=0.9)
    ax[2].set_title("how far each pixel was moved")
    for a in ax:
        a.set_xlabel("longitude [° E]"); a.set_ylabel("latitude [° N]")
    plt.show()

The corrected file has the same pixel grid as the input (`--native`), with the channels TIR1, TIR2, WV and MIR, the cloud-top height `cth`, the shift `shift_km` and a `fill_flag` (0 observed, 1 interpolated, 2 not seen by the satellite). Run the next cell to download it.

The same correction from a terminal, for many files at once:

```bash
pygeoparallax l1c "3RIMG_*_L1C_SGP_V01R00.h5" -o corrected/ --native --fill-empty
```

See the [README](https://github.com/bapallinaveen60/PyGeoParallax-IN#readme) for all options (regular lat/lon grid, terrain, ERA5 cloud heights, navigation correction, a lower effective height for precipitation work).

In [ ]:
if l1c:
    try:
        from google.colab import files
        files.download(f"out/{name}_corrected.nc")
    except ImportError:
        print("corrected file:", f"out/{name}_corrected.nc")